# Clasificación — Regresión Logística

_Sigmoide, probabilidades, frontera de decisión y métricas_

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

![Aprendizaje Supervisado](assets/header.png)

## 1. ¿Qué es la clasificación?

Predecir una etiqueta categórica:

- **Binaria**: $y \in \{0, 1\}$ — por ejemplo, _"¿el cliente se va o se queda?"_
- **Multiclase**: $y \in \{1, \dots, K\}$ — por ejemplo, tipo de producto.

Como la salida es discreta, no podemos usar regresión lineal directamente: nos interesa estimar una **probabilidad** entre 0 y 1.

## 2. La función sigmoide

Para que un modelo lineal entregue probabilidades válidas, pasamos su salida por la **sigmoide**, que aplasta cualquier número real al rango $(0, 1)$:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 3))
plt.plot(z, 1 / (1 + np.exp(-z)), lw=2)
plt.axhline(0.5, color='red', ls='--', alpha=0.6)
plt.axvline(0,   color='gray', ls=':',  alpha=0.6)
plt.title('Sigmoide:  σ(z) = 1 / (1 + e^-z)')
plt.xlabel('z'); plt.ylabel('σ(z)')
plt.grid(True); plt.show()

## 3. El modelo de regresión logística

Aplicamos la sigmoide a una combinación lineal de los predictores:

$$
P(y = 1 \mid x) = \sigma\!\left(\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p\right)
$$

Es una **regresión lineal con un "sombrero" sigmoide** que la convierte en probabilidad. El modelo se ajusta buscando los $\beta$ que hagan que las probabilidades predichas se parezcan lo más posible a las etiquetas reales (internamente minimiza la **log-loss**; para nosotros lo importante es que `scikit-learn` lo hace en una línea).

## 4. Frontera de decisión

Con un umbral típico de 0.5, clasificamos como positivo si $\hat{p} \ge 0.5$. En el espacio de los predictores eso es una **línea recta** (o hiperplano): por eso la regresión logística genera **fronteras lineales**.

El umbral se puede mover según el costo de los errores: bajarlo aumenta el recall (no perder positivos), subirlo aumenta la precisión.

## 5. Métricas de clasificación

**Matriz de confusión**:

|              | Pred. positivo | Pred. negativo |
|---|---|---|
| **Real positivo** | TP | FN |
| **Real negativo** | FP | TN |

$$
\text{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}
\qquad
\text{Precision} = \frac{TP}{TP+FP}
\qquad
\text{Recall} = \frac{TP}{TP+FN}
$$

$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

- **Accuracy** — qué fracción acertamos. Engaña con clases desbalanceadas.
- **Precision** — de los que dije positivos, ¿cuántos lo eran?
- **Recall** — de los positivos reales, ¿cuántos detecté?
- **F1** — promedio armónico de precision y recall.
- **ROC-AUC** — área bajo la curva TPR vs FPR al variar el umbral. AUC = 1 perfecto, AUC = 0.5 aleatorio.

## 6. Caso práctico — Telco Customer Churn (Kaggle)

**Dataset:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn  
**Archivo:** `WA_Fn-UseC_-Telco-Customer-Churn.csv` (7.043 × 21).

**Estructura.** Cada fila es un cliente. Las 21 columnas se agrupan así:

- **Identificador**: `customerID`.
- **Demografía**: `gender`, `SeniorCitizen` (0/1), `Partner`, `Dependents`.
- **Cuenta**: `tenure` (meses con la compañía), `Contract` (Month-to-month / One year / Two year), `PaperlessBilling`, `PaymentMethod`.
- **Servicios**: `PhoneService`, `MultipleLines`, `InternetService` (DSL / Fiber / No), `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`.
- **Facturación**: `MonthlyCharges` (USD/mes), `TotalCharges` (USD acumulado — viene como string con espacios para clientes nuevos, hay que convertir a numérico).
- **Target**: `Churn` (Yes / No).

**Problema:** clasificación binaria — predecir si el cliente se va a dar de baja. Hay desbalance moderado (≈26.5% churners). El loader ya:
1. convierte `TotalCharges` a numérico y descarta las filas con NaN,
2. mapea `Churn` a 1/0.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

df = pd.read_csv(DATA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print('Shape:', df.shape, '| Tasa de churn:', round(df.Churn.mean(), 3))
df.head()

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)
y = df['Churn']
print('Shape de X:', X.shape)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

logit = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)

coefs = pd.DataFrame({
    'feature': X.columns,
    'beta': logit.coef_[0],
}).sort_values('beta', key=abs, ascending=False).head(12)
coefs

**Cómo leer los coeficientes (intuición sin entrar en log-odds):**
- `beta` positivo → la variable **empuja la probabilidad de churn hacia arriba**.
- `beta` negativo → empuja la probabilidad **hacia abajo**.
- Magnitud → cuánto. Como estandarizamos los predictores, las magnitudes son comparables entre sí.

In [ ]:
y_pred = logit.predict(X_te_s)
y_proba = logit.predict_proba(X_te_s)[:, 1]

print(f'Accuracy  : {accuracy_score(y_te, y_pred):.3f}')
print(f'Precision : {precision_score(y_te, y_pred):.3f}')
print(f'Recall    : {recall_score(y_te, y_pred):.3f}')
print(f'F1        : {f1_score(y_te, y_pred):.3f}')
print(f'ROC AUC   : {roc_auc_score(y_te, y_proba):.3f}')

print('\n', classification_report(y_te, y_pred, target_names=['No churn', 'Churn']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred),
    display_labels=['No churn', 'Churn'],
).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Matriz de confusión')

RocCurveDisplay.from_predictions(y_te, y_proba, ax=axes[1])
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('Curva ROC')
plt.tight_layout(); plt.show()

In [ ]:
# --- Distribución de probabilidades por clase real ---
fig, ax = plt.subplots(figsize=(7, 4))
for label, name in [(0, 'No churn'), (1, 'Churn')]:
    ax.hist(y_proba[y_te == label], bins=30, alpha=0.6, label=name)
ax.axvline(0.5, color='red', ls='--', label='umbral 0.5')
ax.set_xlabel('P(churn) predicha'); ax.set_ylabel('frecuencia')
ax.set_title('Probabilidades predichas por clase real')
ax.legend(); plt.show()

## 7. Referencias

- ISLR cap. 4: *Classification*.
- Hosmer, Lemeshow & Sturdivant (2013). *Applied Logistic Regression*.
- scikit-learn user guide: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
- Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn